# BirdieAPI: a season of golf in five minutes

This notebook loads the free BirdieAPI season pack (the last completed season, as CSV), answers a few questions offline, then asks the live API the same kind of question with a free key.

Data: BirdieAPI (birdieapi.com), under the [BirdieAPI Sample Data Licence](https://github.com/sports-data-hq/birdieapi-mcp/blob/main/DATA-LICENSE.md).

In [ ]:
import io, zipfile, urllib.request
import pandas as pd

URL = 'https://github.com/sports-data-hq/birdieapi-mcp/releases/download/season-pack/birdieapi-season-pack.zip'
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
tables = {n[:-4]: pd.read_csv(z.open(n)) for n in z.namelist() if n.endswith('.csv')}

def t(league, kind):
    """One table, e.g. t('pga', 'players')."""
    return next(df for name, df in tables.items() if f'-{league}-' in name and name.endswith('-' + kind))

sorted(tables)[:12]

## Strokes versus the field, adjusted for field strength, minimum 40 rounds

In [ ]:
g = t('pga', 'players')
g[g.rounds >= 40].sort_values('adj_strokes_vs_field_per_round', ascending=False)[['player', 'events', 'wins', 'rounds', 'adj_strokes_vs_field_per_round']].head(10)

## Who won

In [ ]:
r = t('pga', 'tournaments')
r.head(10)

## Scoring average against strokes versus the field

In [ ]:
g = t('pga', 'players')
g = g[g.rounds >= 40]
ax = g.plot.scatter(x='scoring_average', y='adj_strokes_vs_field_per_round', figsize=(7, 6), title='Scoring average vs adjusted strokes vs field')
for _, r in g.iterrows():
    ax.annotate(r['player'], (r['scoring_average'], r['adj_strokes_vs_field_per_round']), fontsize=7)

## The same, live

Every earlier season, play-by-play, lineups, odds and today's games are on the BirdieAPI API. Get a free key and paste it below: one key for every sport at [app.sportsdatahq.com](https://app.sportsdatahq.com/signup?utm_source=colab&utm_medium=notebook&utm_campaign=birdieapi-notebook) (1,000 credits a month), which this notebook uses.

In [ ]:
import getpass, json, urllib.request
KEY = getpass.getpass('API key: ')
req = urllib.request.Request('https://api.sportsdatahq.com/v1/golf/get_ratings', data=b'{}',
    headers={'x-api-key': KEY, 'content-type': 'application/json'})
res = urllib.request.urlopen(req)
print('credits used:', res.headers.get('x-credits-used'), '| left:', res.headers.get('x-credits-remaining'))
json.loads(res.read())['data']

Ask the same questions in plain English: add the MCP server `https://mcp.sportsdatahq.com/mcp` to Claude or ChatGPT, or install the golf skills from https://github.com/sports-data-hq/birdieapi-mcp.